In [1]:
import pandas as pd

df = pd.read_csv("../../../data/bank-marketing/bank-full.csv", sep=";")
print(df.head())

print(df.shape)

assert "y" in df.columns

df.dtypes

   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  
(45211, 17)


age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [2]:
#Converts "yes" to 1 & "no" to 0
assert df.y.isin(["yes", "no"]).all()
print(df.y.unique())
mapping = {"yes": 1, "no": 0}
y = df["y"].map(mapping)

#Displays number of observations in each class
print(y.value_counts())

#Displays the percentage of observations in each class
print(y.value_counts(normalize=True) * 100)

#Confirms there are no missing values
assert y.notna().all(axis=None)

<StringArray>
['no', 'yes']
Length: 2, dtype: str
y
0    39922
1     5289
Name: count, dtype: int64
y
0    88.30152
1    11.69848
Name: proportion, dtype: float64


In [3]:
X = df.drop(columns=["y", "duration"])

assert "y" not in X.columns and "duration" not in X.columns

print(X.shape)
print(X.columns)

(45211, 15)
Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous',
       'poutcome'],
      dtype='str')


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)



In [5]:
print(X_train.shape, X_temp.shape, y_train.shape, y_temp.shape)

print(y_train.value_counts(normalize=True))

print(y_temp.value_counts(normalize=True))

assert X_train.shape[0] == y_train.shape[0]
assert X_temp.shape[0] == y_temp.shape[0]

assert X_train.shape[0] + X_temp.shape[0] == df.shape[0]

(27126, 15) (18085, 15) (27126,) (18085,)
y
0    0.883027
1    0.116973
Name: proportion, dtype: float64
y
0    0.882997
1    0.117003
Name: proportion, dtype: float64


In [6]:
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.value_counts(normalize=True), y_val.value_counts(normalize=True), y_test.value_counts(normalize=True))

(27126, 15) (9042, 15) (9043, 15)
y
0    0.883027
1    0.116973
Name: proportion, dtype: float64 y
0    0.88299
1    0.11701
Name: proportion, dtype: float64 y
0    0.883003
1    0.116997
Name: proportion, dtype: float64


In [7]:
import numpy as np
from sklearn.metrics import accuracy_score

majority_class = y_train.value_counts().idxmax()
print(f"Majority Class: {majority_class}")
assert majority_class in [0, 1]

y_val_pred = np.full(X_val.shape[0], majority_class)
print(y_val_pred[:10])

assert len(y_val_pred) == len(y_val)

assert np.all(y_val_pred == majority_class)

baseline_accuracy = round(accuracy_score(y_val, y_val_pred) * 100, 2)
print(f"value:{baseline_accuracy}%")

calculated_accuracy = round((y_val == 0).sum() / len(y_val) * 100, 2)

assert np.isclose(baseline_accuracy, calculated_accuracy)

Majority Class: 0
[0 0 0 0 0 0 0 0 0 0]
value:88.3%


In [8]:
assert len(X_train.index.intersection(X_val.index)) == 0
assert len(X_test.index.intersection(X_val.index)) == 0
assert len(X_train.index.intersection(X_test.index)) == 0

## Day 15 — Bank Subscription Classification Results

### 1. Problem Definition

The objective is to develop a binary classification system that predicts whether a bank customer will subscribe to a term deposit following a marketing campaign.

Predictions are intended to occur before the current marketing phone call.

Target encoding:
- `0`: Customer did not subscribe.
- `1`: Customer subscribed.

The eventual goal is to identify customers who are more likely to subscribe, helping the bank prioritize marketing efforts.

### 2. Dataset Overview

**Source:** UCI Bank Marketing Dataset (`bank-full.csv`)

- Total observations: 45,211
- Original columns: 17
- Input features after exclusions: 15
- Target: `y`
- File delimiter: Semicolon (`;`)

**Target distribution:**

| Class | Observations | Percentage |
|---|---:|---:|
| 0 — No subscription | 39,922 | 88.30% |
| 1 — Subscription | 5,289 | 11.70% |

The dataset is imbalanced, with significantly fewer positive observations than negative observations.

### 3. Data Preparation and Leakage Prevention

The target was validated against the expected categories (`yes`, `no`) and encoded using an explicit mapping dictionary.

Two columns were excluded from the feature matrix:

- `y`: The prediction target.
- `duration`: The duration of the current marketing call, which is unavailable before the call occurs.

The original DataFrame was preserved.

Assertions verified the target encoding, feature exclusions, and absence of missing encoded labels.

### 4. Stratified Dataset Splitting

Used scikit-learn's `train_test_split()` with `random_state=42` and stratification.

Splitting occurred in two stages:

1. Original dataset → 60% training and 40% temporary.
2. Temporary dataset → 50% validation and 50% test.

**Final partitions:**

| Partition | Observations | Percentage |
|---|---:|---:|
| Training | 27,126 | 60% |
| Validation | 9,042 | 20% |
| Test | 9,043 | 20% |
| Total | 45,211 | 100% |

All three partitions preserved approximately 88.3% non-subscribers and 11.7% subscribers.

Assertions verified:
- Matching feature and target lengths.
- Total observation counts.
- No original row-index overlap across partitions.

**Limitation:** Random stratification does not establish chronological independence. Future experiments should consider time-based splitting to simulate prediction on later marketing campaigns. Index-disjoint partitions also do not guarantee customer-level independence or the absence of duplicate records.

### 5. Majority-Class Baseline

The majority class was determined exclusively from the training target using `value_counts().idxmax()`.

**Majority class:** `0` (No subscription)

A NumPy array was generated to predict the majority class for every validation observation.

Validation accuracy was calculated using scikit-learn's `accuracy_score()` and independently checked against the proportion of non-subscribers in validation.

**Validation accuracy: 88.30%**

No machine learning estimator was trained during this session.

### 6. Baseline Limitations

Although the baseline achieves 88.30% accuracy, it never predicts a positive subscription.

Consequently:
- It identifies zero actual subscribers.
- It has zero recall for the positive class.
- Accuracy alone is insufficient to evaluate its business usefulness.

Future classifiers should be compared using metrics such as precision, recall, F1-score, and ROC AUC, alongside accuracy.

### 7. Engineering Verification

Completed checks included:

- Dataset parsing and column verification.
- Binary target validation.
- Feature exclusion assertions.
- Stratified class-distribution inspection.
- Partition size and alignment assertions.
- Pairwise row-index overlap checks.
- Prediction-length and prediction-value assertions.
- A mutation test confirming that incorrect predictions are detected.
- Independent verification of baseline accuracy.

The test partition was reserved for later model evaluation. Its class distribution was inspected, but no test-set performance metric was calculated or used for model selection.

### 8. Conclusion

Day 15 established the foundation for a reproducible bank subscription classification workflow.

The central finding is that class imbalance can produce deceptively high classification accuracy. The 88.30% majority-class baseline provides a reference against which future classification models can be evaluated.

**Next:** Develop a classification model that can identify positive subscriptions and evaluate its performance beyond accuracy.

## Day 16 — Mixed-Feature Preprocessing

In [9]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()
print(numeric_features)
print(len(numeric_features))
print(categorical_features)
print(len(categorical_features))

assert set(numeric_features).isdisjoint(set(categorical_features))
assert set(numeric_features) | set(categorical_features) == set(X_train.columns)

assert "duration" not in numeric_features and "y" not in numeric_features
assert "duration" not in categorical_features and "y" not in categorical_features

assert (len(numeric_features) + len(categorical_features)) == 15

['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
6
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
9


/tmp/ipykernel_2122/238417453.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include="object").columns.tolist()


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

preprocessing_pipeline = Pipeline(
    [("preprocessing", preprocessor)]
)

print(type(preprocessing_pipeline.steps))
print(type(numeric_features))
print(numeric_features)

assert isinstance(numeric_features, list)
assert isinstance(categorical_features, list)
assert len(numeric_features) == 6
assert len(categorical_features) == 9

<class 'list'>
<class 'list'>
['age', 'balance', 'day', 'campaign', 'pdays', 'previous']


In [11]:
preprocessing_pipeline.fit(X_train)
X_train_processed = preprocessing_pipeline.transform(X_train)
X_val_processed = preprocessing_pipeline.transform(X_val)

print(X_train_processed.shape)
print(X_val_processed.shape)

(27126, 50)
(9042, 50)


In [12]:
X_val_unknown = X_val.copy()
assert X_val.equals(X_val_unknown)

X_val_unknown.loc[X_val_unknown.index[0], "job"] = "astronaut"
X_val_unknown_processed = preprocessing_pipeline.transform(X_val_unknown)

assert X_val_unknown.loc[X_val_unknown.index[0], "job"] == "astronaut"

assert X_val_unknown_processed.shape == X_val_processed.shape

In [13]:
feature_names = preprocessing_pipeline.get_feature_names_out()
cat_job = [name for name in feature_names if name.startswith("cat__job_")]
print(cat_job)
print(len(cat_job))

['cat__job_admin.', 'cat__job_blue-collar', 'cat__job_entrepreneur', 'cat__job_housemaid', 'cat__job_management', 'cat__job_retired', 'cat__job_self-employed', 'cat__job_services', 'cat__job_student', 'cat__job_technician', 'cat__job_unemployed', 'cat__job_unknown']
12


In [14]:
job_indices = []

for i, name in enumerate(feature_names):
    if feature_names[i].startswith("cat__job_"):
        print(i, name)
        job_indices.append(i)

assert len(job_indices) == 12
print(job_indices)

job_values = X_val_unknown_processed[0, job_indices]

assert np.all(job_values == 0)
# False

6 cat__job_admin.
7 cat__job_blue-collar
8 cat__job_entrepreneur
9 cat__job_housemaid
10 cat__job_management
11 cat__job_retired
12 cat__job_self-employed
13 cat__job_services
14 cat__job_student
15 cat__job_technician
16 cat__job_unemployed
17 cat__job_unknown
[6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


## Day 16 — Mixed-Feature Preprocessing Results

### Objective
Build a reusable scikit-learn preprocessing pipeline that handles numerical and categorical features, prevents preprocessing leakage, and supports unseen categories during inference.

### Implementation
- Identified **6 numerical features** and **9 categorical features** using Pandas data types.
- Constructed a `ColumnTransformer` that passes numerical features through unchanged and applies `OneHotEncoder(handle_unknown="ignore")` to categorical features.
- Wrapped the transformer in a scikit-learn `Pipeline`.
- Fitted the pipeline exclusively on `X_train`.
- Transformed both training and validation data using the fitted pipeline.
- Preserved the exclusion of `duration` because it is unavailable before the marketing call.

### Results

| Dataset | Original shape | Transformed shape |
|---|---|---|
| Training | (27,126, 15) | (27,126, 50) |
| Validation | (9,042, 15) | (9,042, 50) |

The 50 transformed features consist of 6 numerical features and 44 one-hot-encoded categorical features.

### Verification
- Confirmed that numerical and categorical feature groups are disjoint and collectively cover all 15 input features.
- Confirmed that feature selectors contain column names rather than DataFrames.
- Confirmed the pipeline produces consistent feature dimensions.
- Tested an unseen `job="astronaut"` category without refitting the pipeline.
- Verified that all 12 encoded job indicators are zero for the unseen category.
- Replaced hardcoded column positions with dynamically identified feature indices.
- Successfully restarted the Jupyter kernel and executed all cells.

### Engineering Lessons
- `.fit()` learns preprocessing parameters; `.transform()` applies them without relearning.
- `handle_unknown="ignore"` prevents errors for unseen categories but does not create new feature columns.
- An unseen category is not automatically mapped to a literal `"unknown"` category learned during training.
- Pandas chained assignment can silently invalidate intended test modifications.
- Passing assertions does not guarantee that a test verifies its intended behavior.
- Jupyter execution order and inconsistent variable names can hide reproducibility defects.

### Limitations and Next Steps
- Numerical features are currently passed through without scaling, imputation, or special handling of sentinel values such as `pdays=-1`.
- No classifier has been trained yet.
- No positive-class precision, recall, F1, or ROC AUC has been measured.
- The final test dataset remains reserved for later evaluation.

**Status:** Day 16 core objectives completed.

Day 17 — Logistic Regression

In [15]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print(sigmoid(np.array([-2, 0, 2])))

z = np.array([-10, -2, 0, 2, 10])
outputs = sigmoid(z)

assert np.isclose(sigmoid(0), 0.5)
assert np.isclose(sigmoid(-2), (1 - sigmoid(2)))
assert np.all((outputs >= 0) & (outputs <= 1))

print(sigmoid(1000))
print(sigmoid(-1000))

[0.11920292 0.5        0.88079708]
1.0
0.0


/tmp/ipykernel_2122/3477629900.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


In [16]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train_processed, y_train)



/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [17]:
X_train[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
age,27126.0,40.852393,10.636893,18.0,33.0,39.0,48.0,95.0
balance,27126.0,1361.175035,3053.628721,-8019.0,74.0,449.0,1425.0,102127.0
day,27126.0,15.781132,8.320549,1.0,8.0,16.0,21.0,31.0
campaign,27126.0,2.750276,3.063629,1.0,1.0,2.0,3.0,63.0
pdays,27126.0,39.928519,99.831751,-1.0,-1.0,-1.0,-1.0,871.0
previous,27126.0,0.582393,2.572319,0.0,0.0,0.0,0.0,275.0


In [18]:
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

preprocessing_pipeline = Pipeline(
    [("preprocessing", preprocessor)]
)

preprocessing_pipeline.fit(X_train)
X_train_processed = preprocessing_pipeline.transform(X_train)
X_val_processed = preprocessing_pipeline.transform(X_val)
print(X_train_processed.shape)
print(X_val_processed.shape)

(27126, 50)
(9042, 50)


In [19]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_processed, y_train)

print(logistic_model.n_iter_)

y_val_pred = logistic_model.predict(X_val_processed)
print(y_val_pred[:11])
print(np.unique(y_val_pred))

[33]
[0 0 0 0 0 0 0 0 0 0 0]
[0 1]


In [20]:
correct = np.sum(y_val_pred == y_val)
logistic_accuracy = correct / y_val_pred.shape[0]
print(logistic_accuracy)

0.8939393939393939


In [21]:
true_positives = np.sum((y_val_pred == 1) & (y_val == 1))
false_negatives = np.sum((y_val_pred == 0) & (y_val == 1))

print(true_positives, false_negatives)

recall = true_positives/(true_positives + false_negatives)

print(f"Recall: {recall}")

185 873
Recall: 0.17485822306238186


In [22]:
y_val_proba = logistic_model.predict_proba(X_val_processed)

print(y_val_proba.shape)

y_val_proba[:5]

(9042, 2)


array([[0.93102833, 0.06897167],
       [0.88489608, 0.11510392],
       [0.91346237, 0.08653763],
       [0.92134224, 0.07865776],
       [0.90600978, 0.09399022]])

In [23]:
decision_scores = logistic_model.decision_function(X_val_processed)

print(decision_scores[:5])

[-2.60259391 -2.03963483 -2.35666286 -2.46072525 -2.26585939]


In [24]:
manual_probabilities = sigmoid(decision_scores)

print(manual_probabilities[:5])

assert np.allclose(manual_probabilities, y_val_proba[:,1])

[0.06897167 0.11510392 0.08653763 0.07865776 0.09399022]


## Day 17 — Logistic Regression

### Objective

Implement the sigmoid function with NumPy, train a logistic regression classifier on the Bank Marketing dataset, evaluate it against the majority-class baseline, and understand the relationship between decision scores and predicted probabilities.

### Implementation

**Sigmoid function**

Implemented the mathematical sigmoid function using NumPy's vectorized exponential operation.

Verified:
- Sigmoid of zero equals 0.5.
- Sigmoid symmetry.
- Outputs remain between zero and one for the tested moderate inputs.
- Extreme inputs can produce floating-point overflow warnings or rounded boundary values.

**Preprocessing**

Updated the existing ColumnTransformer to use StandardScaler for six numerical features and OneHotEncoder for nine categorical features.

The preprocessing pipeline was fitted exclusively on training data and then applied to training and validation datasets.

Transformed shapes:
- Training: (27126, 50)
- Validation: (9042, 50)

**Model training**

Trained scikit-learn LogisticRegression with max_iter=1000.

The initial unscaled model reached the iteration limit without converging. After standardization, the model converged in 33 iterations.

### Validation results

| Metric | Result |
|---|---|
| Majority-class accuracy | Approximately 88.30% |
| Logistic regression accuracy | 89.39% |
| Accuracy improvement | Approximately 1.09 percentage points |
| True positives | 185 |
| False negatives | 873 |
| Recall | 17.49% |

The model improved validation accuracy compared with the majority-class baseline but missed most actual subscribers.

### Probability verification

Used decision_function() to obtain raw model scores.

Applied the manually implemented sigmoid function to these scores and compared the results against the positive-class output from predict_proba().

The np.allclose() assertion passed across the complete validation dataset.

### Engineering lessons

- Logistic regression transforms linear decision scores into estimated class probabilities using sigmoid.
- Feature scaling can significantly improve numerical optimization.
- Training-derived preprocessing parameters must be reused on validation data to prevent leakage.
- Accuracy alone is insufficient for evaluating imbalanced classification problems.
- Recall measures the proportion of actual positive observations correctly identified.
- Classification thresholds affect the balance between false positives and false negatives.

### Limitations and next steps

- Sigmoid has not been made numerically stable for extreme inputs.
- Precision, F1, ROC-AUC, and probability calibration have not been evaluated.
- No classification threshold optimization has been performed.
- Final test data remains untouched.
- Fresh-kernel reproducibility verification is pending.

## Day 18 - Build Confusion Matrix

In [36]:
def confusion_matrix_manual(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_pred == 1) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))

    confusion_matrix = np.array([
        [tn, fp],
        [fn, tp]
    ])

    return confusion_matrix

y_true = [1, 0, 1, 0, 1, 0]
y_pred = [1, 1, 0, 0, 1, 0]

print(confusion_matrix_manual(y_true, y_pred))

y_true = [1, 1, 0, 0, 0]
y_pred = [1, 0, 0, 0, 0]

answer = np.array([
    [3, 0],
    [1, 1]
])

assert np.allclose(confusion_matrix_manual(y_true, y_pred), answer)
print(confusion_matrix_manual(y_true, y_pred))


[[2 1]
 [1 2]]
[[3 0]
 [1 1]]


In [37]:
from sklearn.metrics import confusion_matrix

assert np.array_equal(confusion_matrix(y_true, y_pred), confusion_matrix_manual(y_true, y_pred))

print(confusion_matrix(y_true, y_pred))
print(confusion_matrix_manual(y_true, y_pred))


[[3 0]
 [1 1]]
[[3 0]
 [1 1]]


In [38]:
y_true = [0, 0, 0, 1, 1]
y_pred = [0, 0, 0, 0, 0]

assert np.array_equal(confusion_matrix(y_true, y_pred, labels=[0, 1]), confusion_matrix_manual(y_true, y_pred))

print(confusion_matrix(y_true, y_pred))
print(confusion_matrix_manual(y_true, y_pred))


[[3 0]
 [2 0]]
[[3 0]
 [2 0]]


In [42]:
y_true = [0, 0, 0]
y_pred = [0, 0, 0]

print(confusion_matrix(y_true, y_pred).shape)
print(confusion_matrix_manual(y_true, y_pred).shape)

(1, 1)
(2, 2)


/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [43]:
assert np.array_equal(confusion_matrix(y_val, y_val_pred, labels=[0, 1]), confusion_matrix_manual(y_val, y_val_pred))

print(confusion_matrix(y_val, y_val_pred))
print(confusion_matrix_manual(y_val, y_val_pred))

[[7898   86]
 [ 873  185]]
[[7898   86]
 [ 873  185]]


# Day 18 — Confusion Matrix From Scratch

## Objective

Implement a binary classification confusion matrix using NumPy, understand true positives, true negatives, false positives, and false negatives, and verify the implementation against scikit-learn.

**Project:** Bank Subscription Classification  
**Dataset:** UCI Bank Marketing  
**Model:** Logistic Regression  
**Validation observations:** 9,042

## 1. Confusion Matrix Overview

A confusion matrix evaluates classification predictions by separating correct and incorrect predictions into four categories.

| | Predicted 0 | Predicted 1 |
|---|---|---|
| Actual 0 | True Negative (TN) | False Positive (FP) |
| Actual 1 | False Negative (FN) | True Positive (TP) |

For this project:

- **True Positive (TP):** Correctly predicts that a customer will subscribe.
- **True Negative (TN):** Correctly predicts that a customer will not subscribe.
- **False Positive (FP):** Predicts subscription when the customer does not subscribe.
- **False Negative (FN):** Predicts no subscription when the customer actually subscribes.

## 2. Implementation

Implemented `confusion_matrix_manual(y_true, y_pred)` independently using NumPy.

The function:

1. Converts both inputs into NumPy arrays.
2. Uses element-wise Boolean comparisons to count TP, TN, FP, and FN.
3. Uses `np.sum()` to aggregate the counts.
4. Constructs a 2×2 NumPy array in the format `[[TN, FP], [FN, TP]]`.
5. Returns the resulting confusion matrix.

The implementation does not depend on scikit-learn for its calculations.

## 3. Testing and Verification

The implementation was tested using several scenarios.

| Test | Result |
|---|---|
| Six-customer manual calculation | Passed |
| Unequal TP and TN values | Passed |
| Comparison against scikit-learn | Passed |
| Classifier predicting only class 0 | Passed |
| Dataset containing only class 0 | Passed |
| Full bank validation dataset | Passed |

Exact equality was verified using `np.array_equal()`.

The implementation was compared with scikit-learn's `confusion_matrix()` using `labels=[0, 1]` to ensure a consistent 2×2 matrix.

## 4. Validation Results

The trained logistic regression model from Day 17 was evaluated on 9,042 validation observations.

**Confusion matrix:**

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | 7,898 | 86 |
| Actual 1 | 873 | 185 |

**Performance:**

| Metric | Value |
|---|---:|
| True Positives | 185 |
| True Negatives | 7,898 |
| False Positives | 86 |
| False Negatives | 873 |
| Correct predictions | 8,083 |
| Incorrect predictions | 959 |
| Validation accuracy | 89.39% |
| Positive-class recall | 17.49% |
| Majority-class baseline accuracy | Approximately 88.30% |

The logistic regression model improves accuracy by approximately 1.09 percentage points over the majority-class baseline.

However, it correctly identifies only 185 of 1,058 actual subscribers, missing 873.

This demonstrates why accuracy alone is insufficient for evaluating an imbalanced classification problem.

## 5. Debugging and Engineering Lessons

**Python lists versus NumPy arrays**

Python lists do not perform element-wise comparisons with integers. Converting inputs to NumPy arrays enables the Boolean comparisons required to calculate confusion matrix counts.

**Matrix ordering**

The conventional scikit-learn binary confusion matrix ordering is `[[TN, FP], [FN, TP]]`.

An initial mistake placed TP where TN belonged. The first test did not reveal this because both values were equal. A second test with unequal values exposed the issue.

**Absent classes**

When only class 0 appears in the dataset, scikit-learn can return a 1×1 confusion matrix unless the labels are explicitly specified.

Using `labels=[0, 1]` produces a consistent 2×2 matrix.

**Exact testing**

`np.array_equal()` is preferred over `np.allclose()` when comparing integer confusion matrices, because exact counts should match.

## 6. Limitations and Future Improvements

The current function assumes binary labels 0 and 1.

Potential improvements include:

- Validate that input arrays have equal lengths.
- Reject labels other than 0 and 1.
- Handle empty inputs and invalid array dimensions.
- Add automated unit tests.
- Verify reproducibility through a complete fresh-kernel notebook execution.

Invalid labels currently risk being silently excluded from the four counts.

These improvements are optional enhancements rather than requirements for the completed Day 18 lesson.

## 7. Conclusion

The confusion matrix was successfully implemented from scratch and verified against scikit-learn using both synthetic examples and the real bank validation dataset.

The logistic regression model achieves 89.39% accuracy but only 17.49% positive-class recall, indicating that it fails to identify most customers who actually subscribe.

**Definition of done: Completed.**

**Next — Day 19:** Calculate precision, recall, and F1 score, then investigate how probability thresholds affect false positives, false negatives, and business outcomes.

The final test dataset remains untouched.